In [ ]:
import cv2
import numpy as np
import mediapipe as mp
import time
import math
from mediapipe.tasks import python
from mediapipe.tasks.python import vision


base_options = python.BaseOptions(model_asset_path="src/models/MediaPipe/hand_landmarker.task")
options = vision.HandLandmarkerOptions(base_options=base_options, num_hands=2)
hand_detector = vision.HandLandmarker.create_from_options(options)
connections = vision.HandLandmarksConnections.HAND_CONNECTIONS

# 손가락 끝 Landmark index와 이름 매핑
finger_tips = {
    4: "엄지", 
    8: "검지", 
    12: "중지", 
    16: "약지", 
    20: "소지"
}


green_lower = np.array([30, 60, 90], dtype=np.uint8)
green_upper = np.array([230, 115, 180], dtype=np.uint8)
kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))


pipeline = (
    "nvarguscamerasrc sensor-id=0 ! "
    "video/x-raw(memory:NVMM), width=1280, height=720, framerate=30/1 ! "
    "nvvidconv ! "
    "video/x-raw, format=BGRx ! "
    "videoconvert ! "
    "video/x-raw, format=BGR ! "
    "queue leaky=downstream max-size-buffers=1 ! "
    "appsink drop=true max-buffers=1 sync=false"
)

cap = cv2.VideoCapture(pipeline, cv2.CAP_GSTREAMER)

while True:
    ret, frame = cap.read()
    if not ret:
        break
        
    frame = cv2.flip(frame, 0)
    h, w = frame.shape[:2]

    
    blr = cv2.GaussianBlur(frame, (11, 11), 0)
    lab = cv2.cvtColor(blr, cv2.COLOR_BGR2LAB)
    mask = cv2.inRange(lab, green_lower, green_upper)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel, iterations=2)
    mask = cv2.dilate(mask, kernel, iterations=2)
    
    contour_lst, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    ball_center = None
    if len(contour_lst) > 0:
        contour = max(contour_lst, key=cv2.contourArea)
        _, radius = cv2.minEnclosingCircle(contour)
        M = cv2.moments(contour)
        if M["m00"] != 0:
            ball_center = (int(M["m10"] / M["m00"]), int(M["m01"] / M["m00"]))
            # 공 주변에 원 및 중심점 그리기
            cv2.circle(frame, ball_center, int(radius), (255, 0, 0), 2)
            cv2.circle(frame, ball_center, 5, (255, 0, 0), -1)

   
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
    result = hand_detector.detect(mp_image)

    closest_finger = None
    min_distance = float('inf')

    if result.hand_landmarks:
        for hand in result.hand_landmarks:
            # 관절 좌표를 픽셀 단위로 변환
            points = [(int(p.x * w), int(p.y * h)) for p in hand]
            
            # 손 뼈대 그리기 (Skeleton)
            for c in connections:
                cv2.line(frame, points[c.start], points[c.end], (0, 255, 0), 2)
                
            for i, point in enumerate(points):
                color = (0, 0, 255) if i in finger_tips.keys() else (255, 0, 0)
                cv2.circle(frame, point, 6 if i in finger_tips.keys() else 4, color, -1)


        
            if ball_center:
                for tip_idx, finger_name in finger_tips.items():
                    tip_point = points[tip_idx]
                    # 유클리디안 거리 계산: sqrt((x2 - x1)^2 + (y2 - y1)^2)
                    dist = math.hypot(tip_point[0] - ball_center[0], tip_point[1] - ball_center[1])
                    
                    if dist < min_distance:
                        min_distance = dist
                        closest_finger = finger_name


    if ball_center and closest_finger:
        # 손가락과 공 사이의 거리가 특정 픽셀(예: 150px) 이하일 때만 올려져 있다고 판단
        if min_distance < 150: 
            text = f"Ball is on: {closest_finger}"
            cv2.putText(frame, text, (30, 60), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 255, 255), 3)
        else:
            cv2.putText(frame, "Ball detected, but far from hand", (30, 60), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)
    elif ball_center and not result.hand_landmarks:
        cv2.putText(frame, "Ball detected (No hands)", (30, 60), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)

    cv2.imshow("Hand & Ball Detection", frame)
    
    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()

FileNotFoundError: Unable to open file at src/models/MediaPipe/hand_landmarker.task